In [1]:
# today we look at error nodes - API timeout fallback to local llama

""" 
Today i have to make my graph resilient to the thing that will definitely happen in production
the primary LLM API goes down, times out, or rate-limits you mid run. Right now, if 
Groq throws an exception my entire graph crashes and the checkpoint is useless. By the end of today,
any failure in the primary Groq API automatically reroutes to a local Ollama-served Llama 3 model,
the error is logged into state, and the run completes successfully on the fallback - the user never sees a crash.

Deep coding (Error detection + fallback Routing)
Error-Aware Generate Node: Wrap your generate_node in a try/except block that catches Groq API
exceptions, timeouts, rate limit errors, connection errors, and authentication failures. On any exception, instead of
crashing, the node sets api_error: True in state and returns cleanly
Fallback Generate Node: A second generation node, fallback_generate_node, that is structurally
identical to generate_node but calls local Ollama instance instead of Groq. It also sets
using_fallback_llm: True in state so downstream nodes and your logs know which model produced the answer
Error Log Node: A lightweight error_log_node that sits between a detected failure and then fallback. it appends a structured
entry to the error_log list in state(timestamp, errormessage, which node failed, which fallback was triggered) before the fallback fires.
Conditional Edge After Generate: A new Routing function after generate_node. if api_error is True, route to
error_log_node -> fallback_generate_node -> review_node. if no error, route directly to review_node as before.

"""

' \nToday i have to make my graph resilient to the thing that will definitely happen in production\nthe primary LLM API goes down, times out, or rate-limits you mid run. Right now, if \nGroq throws an exception my entire graph crashes and the checkpoint is useless. By the end of today,\nany failure in the primary Groq API automatically reroutes to a local Ollama-served Llama 3 model,\nthe error is logged into state, and the run completes successfully on the fallback - the user never sees a crash.\n\nDeep coding (Error detection + fallback Routing)\nError-Aware Generate Node: Wrap your generate_node in a try/except block that catches Groq API\nexceptions, timeouts, rate limit errors, connection errors, and authentication failures. On any exception, instead of\ncrashing, the node sets api_error: True in state and returns cleanly\nFallback Generate Node: A second generation node, fallback_generate_node, that is structurally\nidentical to generate_node but calls local Ollama instance inste

In [2]:

# Importing necessary libraries


import os
import re
import json
from groq import Groq
from dataclasses import dataclass, field, asdict
from datetime import datetime
from typing import Optional, TypedDict, Literal
from langgraph.graph import StateGraph, END
from dotenv import load_dotenv, find_dotenv
from dotenv import load_dotenv, find_dotenv
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.mongodb import MongoDBAtlasVectorSearch
from llama_index.storage.docstore.mongodb import MongoDocumentStore
from pymongo import MongoClient, AsyncMongoClient
from llama_index.core import VectorStoreIndex, StorageContext
from langchain_community.tools.tavily_search import TavilySearchResults

Settings.embed_model = HuggingFaceEmbedding(
    model_name = 'BAAI/bge-m3'
)


load_dotenv(find_dotenv())

client = Groq()
webSearch = TavilySearchResults(max_results = 3)


# LOADING EXISTING VECTOR STORE

# first connecting to existing vector store
# connecting to existing vectorstore
# Re-establishing the connection to my mongoDB atlas vector store to retrieve historical data

# mongoDB atlas connection
mongoClient = MongoClient(os.getenv('MONGO_URI'))
asyncMongoClient = AsyncMongoClient(os.getenv('MONGO_URI'))


# reconnecting persistent docstore
docstore = MongoDocumentStore.from_uri(
    uri = os.getenv('MONGO_URI'),
    db_name = 'month1_database',
    namespace = 'month_1_collection'
)

# 1. Connect to your MongoDB
vectorStore = MongoDBAtlasVectorSearch(
    mongodb_client=mongoClient,
    async_mongodb_client= asyncMongoClient, 
    db_name= 'month1_database',
    collection_name= 'month_1_rag_collection_v3',
    vector_index_name= 'final_index_v3',
    embedding_key= 'embedding'
)

storageContext = StorageContext.from_defaults(
    vector_store= vectorStore,
    docstore= docstore
)

index = VectorStoreIndex.from_vector_store(
    vector_store= vectorStore,
    storage_context = storageContext
)

# creating the primary retrieval tool
'''apple_10k_expert = QueryEngineTool(
    query_engine= index.as_query_engine(similarity_top_k = 15),
    metadata= ToolMetadata(
        name = 'apple_10k_expert',
        description= "Search Through Apple's 10-K filings for historical financial data, revenue figures, and risk factors."
    )

)'''

print('🤖🛩️ Vector Store connection established ⚡')


# TOKEN USER TRACKER
@dataclass
class TokenUsage:
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_calls: int = 0

    def add(self, usage):
        self.prompt_tokens += usage.prompt_tokens
        self.completion_tokens += usage.completion_tokens
        self.total_calls += 1
    
    def cost_estimate(
            self,
            input_price_per_1m: float = 2.50,
            output_price_per_1m: float = 10.00
    ) -> float:
        input_cost = (self.prompt_tokens /1000000) * input_price_per_1m
        output_cost = (self.completion_tokens /1000000) * output_price_per_1m
        return round(input_cost + output_cost, 6)
    
    def report(self):
        print(f"\n📣 Token usage report")
        print(f"LLM calls : {self.total_calls}")
        print(f"Prompt tokens: {self.prompt_tokens}")
        print(f"Completion tokens: {self.completion_tokens}")
        print(f"Total tokens: {self.prompt_tokens + self.completion_tokens:,}")
        print(f"Est. cost (GPT-40 pricing): ${self.cost_estimate()}")


# global tracher that is set before each run
usage_tracker = TokenUsage()


# TOKEN aware LLM Caller

def tracked_llm_call(messages: list, system: str = "") -> str:
    """
    Wraps every Groq call so token usage is always captured.
    Drop-in replacement for direct client.chat.completions.create calls.
    
    """

    full_messages = []
    if system:
        full_messages.append({"role": "system", "content": system})
    full_messages.extend(messages)

    response = client.chat.completions.create(
        model = "openai/gpt-oss-120b",
        messages= full_messages,
        temperature= 0,
    )

    usage_tracker.add(response.usage)
    return response.choices[0].message.content

# RETRIEVAL WITH CONFIDENCE SCORING
RELEVANCE_THRESHOLD = 0.5

def retrieve_with_confidence(query: str) -> tuple[list, float]:
    """Returns retrieved docs and the top chunk's confidence score.
    Uses cosine similarity to score from your vectore store.
    """

    # creating a native llamaindex retriever from my initialized index
    retriever = index.as_retriever(similarity_top_k = 4)
    results = retriever.retrieve(query)

    # reults is a list of (document, score) tuples
    # lower score = more similar in FAISS  (L2 distance); invert if needed
    # For Cosine similarity stores, higher = better

    if not results:
        return [], 0.0
    
    
    top_score = float(results[0].score) if results[0].score is not None else 0.0

    print(f"📣 Top retrieval score : {top_score:.3f} (threshold: {RELEVANCE_THRESHOLD})")
    return results, top_score

def format_chunks(docs: list) -> str:
    return "\n\n---\n\n".join([doc.node.get_content() for doc in docs])
        


# CRAG -> Retrieval quality gate

def corrective_retrieve(query: str) -> tuple[str, str]:
    """
    Returns (context_text, source) where source is 'local' or 'web'.
    Applies CRAG logic: low confidence -> discard local, use web fallback.
    """

    docs, top_score = retrieve_with_confidence(query)

    if top_score < RELEVANCE_THRESHOLD or not docs:
        print("🤥 CRAG: Low retrieval confidence - falling back to web search")
        webResults = webSearch.invoke(query)
        context = "\n\n".join([r["content"] for r in webResults])
        return context, "web"
    
    else:
        print("👍 CRAG: Retrieval confidence acceptable - using local docs")
        return format_chunks(docs), "local"
    

GENERATOR_SYSTEM_PROMPT = """You are a precise financial analyst assistant.
Answer the user's question using ONLY the context provided.
If the context does not contain eough information to answer, say exactly:
'I cannot find sufficient information in the provided context.' 
Be specific - include numbners, percentages, and fiscal year references where available.

"""

REVIEWER_SYSTEM_PROMPT = """You are a strict factual reviewer for a financial RAG system.
You will receive a question, the source context, and a generated answer.

Your job is to check:
1. Does the answer contain any claims NOT supported by the context? (hallucination)
2. Does the answer actually address the question asked?
3. Are numbers, percentages, and figures accurate relative to the context?

Respond in EXACTLY this format:
Verdict: <PASS or FAIL>
Reason: <one sentence explaining your verdict>

PASS meaans the answer is faithful to the context and addresses the question,
FAIL means the answer contains unsupported claims, wrong figuress, or avoids the question.


"""

ROUTER_SYSTEM_PROMPT = """You are a query complexity classifier for a financial RAG system.

Classify the user's question as one of:
- SIMPLE: a single factual lookup requiring one retrieval and one answer
(e.g. "What was Apple's net income in FY2024?")
- COMPLEX: requires multiple steps, comparisons, calculations, or chaining
(e.g. "Compare iPhone revenue across FY2023 and FY2024 and calculate the growth rate")
- UNKNOWN: cannot be answered from a financial document at all 
(e.g. "What is the weather in Cupertino today?")

Respond in EXACTLY this format:
Classification: <SIMPLE, COMPLEX, or UNKNOWN>
Reason: <one sentence>
"""

def generate_answer(question: str, context: str, critique: str = "") -> str:
    critiqueBlock = ""
    if critique:
        critiqueBlock = f"\n\nPrevious answer was rejected for this reason: {critique}\nPlease rewrite addressing this critique."

   # response = client.chat.completions.create(
      #  model = "openai/gpt-oss-120b",
        messages = [
            #{"role": "system", "content": GENERATOR_SYSTEM_PROMPT},
            {"role": "user", "content": (
                f"Context:\n{context}\n\n"
                f"Question: {question}"
                f"{critiqueBlock}"
            )}
        ]#,

       # temperature= 0,
   # )

    return tracked_llm_call(messages= messages, system= GENERATOR_SYSTEM_PROMPT)


def review_answer(question: str, context: str, answer: str) -> tuple[str, str]:
    """Returns (verdict, reason) where verdict is a PASS or FAIL."""
    
    messages = [
        #{"role": "system", "content": REVIEWER_SYSTEM_PROMPT},
        {"role": "user", "content":(
            f"Question: {question}\n\n"
            f"Source Context:\n{context}\n\n"
            f"Generated Answer:\n{answer}"

        )}
    ]

    raw = tracked_llm_call(messages= messages, system= REVIEWER_SYSTEM_PROMPT)
    verdict_match = re.search(r"Verdict:\s*(PASS|FAIL)", raw)
    reason_match =re.search(r"Reason:\s*(.+)", raw)

    verdict = verdict_match.group(1) if verdict_match else "FAIL"
    reason = reason_match.group(1).strip() if reason_match else raw.strip()

    print(f"😮‍💨 Reviewer verdict: {verdict} - {reason}")
    return verdict, reason

# NAIVE RAG PATH 

NAIVE_RAG_SYSTEM_PROMPT = """You are a precise financial analyst assistant.
Answer the question uning ONLY THE context provided.
Be specific - include exact figures, percentages, and fiscal year references.
If the context does not contain the answer, say so directly.

"""

def run_naive_rag(question: str) -> dict:
    print("\n⚡ Path: NAIVE RAG")
    started_at = datetime.now().isoformat()

    # single retrieval
    docs, score = retrieve_with_confidence(question)
    context = "\n\n --- \n\n".join([doc.node.get_content() for doc in docs])

    # Single generation - no reviewer, no rewrite
    answer = tracked_llm_call(
        messages=[{
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}"
        }],
        system = NAIVE_RAG_SYSTEM_PROMPT
    )

    print(f"Answer: {answer}")
    return {
        "path": "naive_rag",
        "question": question,
        "answer": answer,
        "retrieval_score": score,
        "started_at": started_at,
        "finished_at": datetime.now().isoformat()
    }


# THE FULL SELF CORRECTING CRAG PIPELINE

def run_crag_pipeline(question: str, max_rewrites: int = 2) ->dict:
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"\n{'='*60}")

    startedAt = datetime.now().isoformat()

    # STEP 1: CRAG retrieval with confidence gate
    context, source = corrective_retrieve(question)

    # STEP 2: Generate Initial Answer
    print("\n 🦾 Generating initial answer...")
    answer = generate_answer(question, context= context)
    print(f"Answer: {answer}\n")

    # SECONDARY GATE to catch false-positive vector scores
    if answer and "I cannot find sufficient information" in answer and source == "local":
        print("🤥 CRAG: Local docs failed to answer despite high vector score. Forcing web fallback...")
        webResults = webSearch.invoke(question)
        context = "\n\n".join([r["content"] for r in webResults])
        source = "web"

        print("👍Generating answer from web context...")
        answer = generate_answer(question, context= context)
        print(f"Web Fallback Answer: {answer}\n")

    # STEP 3: Reviewer Loop
    attempts = 0
    verdict = 'FAIL'
    critique = ""
    history = []

    while verdict == 'FAIL' and attempts < max_rewrites:
        verdict, critique = review_answer(question= question, context= context, answer= answer)
        history.append({
            "attempt": attempts +1,
            "answer": answer,
            "verdict": verdict,
            "critique": critique
        })

        if verdict == 'FAIL':
            attempts += 1
            if attempts < max_rewrites:
                print(f"\n🔁 Rewriting (attempt {attempts})...")
                answer = generate_answer(question, context, critique)
                print(f"Rewritten Answer: {answer}\n")
            else:
                print("🤥 Max rewrites reached - returning best attempt with warning")

    # final evrdict check if we exited the loop with PASS 
    if verdict != 'FAIL':
        verdict, critique = review_answer(question, context, answer)
        history.append({
            "attempt": attempts +1,
            "answer": answer,
            "verdict": verdict,
            "critique": critique
        })

    result = {
        "question": question,
        "retrieval_source": source,
        "final_answer": answer,
        "fianl_verdict": verdict,
        "rewrite_attempts": attempts,
        "review_history": history,
        "started_at":startedAt,
        "finished_at": datetime.now().isoformat()
    }


    print(f"\n{'='*60}")
    print(f"👍 Final Answer ({verdict} after {attempts} rewrite(s)):")
    print(answer)
    print(f"{'='*60}\n")

    filename = f"traces/day11_crag_{datetime.now().strftime('%H%M%S')}.json"
    with open(filename, 'w') as f:
        json.dump(result, f, indent= 2)
    print(f"📀 Saved to {filename}")

    return result


def classify_query(question: str) -> tuple[str, str]:
    raw = tracked_llm_call(
        messages = [{"role": "user", "content": f"Question: {question}"}],
        system= ROUTER_SYSTEM_PROMPT
    )
    classification_match = re.search(r"Classification:\s*(SIMPLE|COMPLEX|UNKNOWN)", raw)
    reason_match = re.search(r"Reason:\s*(.+)", raw)

    classification = classification_match.group(1) if classification_match else "COMPLEX"
    reason = reason_match.group(1).strip() if reason_match else "Could not Parse reason."

    print(f"🦾 Router: {classification} - {reason}")
    return classification, reason

# MULTI-PATH ROUTER

def run_router(question: str) -> dict:
    global usage_tracker
    usage_tracker = TokenUsage() # reset for each question

    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"{'='*60}")

    classification, reason = classify_query(question)

    if classification== 'SIMPLE': 
        result = run_naive_rag(question= question)

    elif classification == "COMPLEX":
        print("\n🤖 Path: AGENTIC RAG (CRAG + Reviewer)")
        result = run_crag_pipeline(question= question)
        result['path'] = "agentic_rag"
    
    else: # UNKNOWN
        print("\n🤥 Path: UNKNOWN - cannot answer from financial documents")
        result = {
            "path": "unknown",
            "question": question,
            "answer": "This question cannot be answered using the Apple 10-K document.",
            "started_at": datetime.now().isoformat(),
            "finished_at": datetime.now().isoformat()
        }
    
    usage_tracker.report()

    result["Classification"] = classification
    result["classification_reason"] = reason
    result["token_usage"] = {
        "prompt_tokens": usage_tracker.prompt_tokens,
        "completion_tokens": usage_tracker.completion_tokens,
        "total_calls": usage_tracker.total_calls,
        "estimated_cost_usd": usage_tracker.cost_estimate()
    }

    filename = f"traces/day12_{classification.lower()}_{datetime.now().strftime('%H%M%S')}.json"
    with open(filename, "w") as f:
        json.dump(result, f, indent = 2)
    print(f"\n📀 Saved to {filename}")

    return result



c:\Users\rodne\miniconda3\envs\rag-Ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\rodne\AppData\Local\Temp\ipykernel_6660\2373246763.py:30: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  webSearch = TavilySearchResults(max_results = 3)


🤖🛩️ Vector Store connection established ⚡


In [3]:
# importing necessary libraries for today

import uuid
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

# improving reused RAG state from day 16

# THE STATE SCHEMA

class RAGState(TypedDict):
    # Input
    question: str

    # Router output
    classification: Optional[str]
    classification_reason: Optional[str]

    # Retrieval output
    context: Optional[str]
    retrieval_source: Optional[str]
    retrieval_score: Optional[float]

    # Generation output
    answer: Optional[str]

    # Reviewer output
    reviewer_verdict: Optional[str]
    reviewer_reason: Optional[str]

    # Rewrite tracking
    rewrite_count: int
    max_rewrites: int

    
    # Sufficiency and reformulation tracking
    sufficiency_verdict: Optional[str]
    sufficiency_reason: Optional[str]
    retrieval_attempts: int
    max_retrieval_attempts: int
    reformulated_query: Optional[str]

    # New today - error tracking
    api_error: bool
    api_error_message: Optional[str]
    api_error_type: Optional[str]
    using_fallback_llm: bool
    error_log: list[dict]

    # Final flags
    warning: Optional[str]
    finished_at: Optional[str]





# THE FIVE NODES

# -----Node 1: Classify -------
def classify_node(state: RAGState) -> dict:
    print(f"\n[NODE: classify] Question: {state['question'][:60]}...")
    raw = tracked_llm_call(
        messages=[{"role": "user", "content": f"Question: {state['question']}"}],
        system= ROUTER_SYSTEM_PROMPT
    )

    classification_match = re.search(r"Classification:\s*(SIMPLE|COMPLEX|UNKNOWN)", raw)
    reason_match = re.search(r"Reason:\s*(.+)", raw)

    classification = classification_match.group(1) if classification_match else "COMPLEX"
    reason = reason_match.group(1).strip() if reason_match else "Could not parse."

    print(f" -> Classification: {classification} - {reason}")
    return {"classification": classification, "classification_reason": reason}

# ------ Node 2: Retrieve ------ # LEAB=VING THIS ONE OUT AS WE ARE GOING TO WRITE A MODIFIED VERSION OF IT IN THE NEXT CELL


# ---Node 3: Generate ----
def generate_node(state: RAGState) -> dict:
    print(f"\n[NODE: generate]")

    if state["classification"] == "UNKNOWN":
        answer = (
            "This question cannot be answered using the Apple FY2024 10-K document"
            "or available tools."
        )
        print(f" -> UNKNOWN path - returning abstention")
        return {"answer": answer}
    
    critique_block = ""
    if state.get("reviewer_reason") and state.get("rewrite_count", 0) > 0:
        critique_block = (
            f"\n\nPrevious answer was rejected: {state['reviewer_reason']}. "
            f"Please rewrite addressing this critique."
        )

    answer = tracked_llm_call(
        messages = [{
            "role":"user",
            "content": (
                f"Context:\n{state['context']}\n\n"
                f"Question: {state['question']}"
                f"{critique_block}"
            )

        }],
        system= GENERATOR_SYSTEM_PROMPT
    )
    print(f" -> Answer generated ({len(answer)} chars)")
    return {"answer": answer}


# -----Node 4: Review ----
def review_node(state: RAGState) -> dict:
    print(f"\n[NODE: review]")

    if state["classification"] == "UNKNOWN":
        print(" -> UNKNOWN path - skipping review")
        return {"reviewer_verdict": "PASS", "reviewer_reason": "Abstention accepted."}
    
    raw = tracked_llm_call(
        messages= [{
            "role": "user",
            "content": (
                f"Question: {state['question']}\n\n"
                f"Source Context:\n{state['context']}\n\n"
                f"Generated Answer:\n{state['answer']}"
            )
        }],
        system = REVIEWER_SYSTEM_PROMPT
    )

    verdict_match =re.search(r"Verdict:\s*(PASS|FAIL)", raw)
    reason_match = re.search(r"Reason:\s*(.+)", raw)

    verdict = verdict_match.group(1) if verdict_match else "FAIL"
    reason = reason_match.group(1).strip() if reason_match else raw.strip()

    print(f" -> Reviewer: {verdict} - {reason}")
    return {"reviewer_verdict" : verdict, "reviewer_reason": reason}


# ---- Node 5: Rewrite ----
def rewrite_node(state: RAGState) -> dict:
    new_count = state.get("rewrite_count", 0) + 1
    print(f"\n[NODE: rewrite] Attempt {new_count}")
    return {
        "rewrite_count" : new_count,
        "answer": None # cleared so generate_node produces a fresh answer
    }


# CONDITIONAL EDGE LOGIC

def route_after_review(state: RAGState) -> Literal["rewrite_node", "__end__"]:
    """
    Called after review_node .
    Decides whether to loop back for a rewrite or proceed to END.

    """

    verdict = state.get("reviewer_verdict", "FAIL")
    rewrite_count = state.get("rewrite_count", 0)
    max_rewrites = state.get("max_rewrites", 2)

    if verdict == "PASS":
        print(" -> Edge: PASS -> END")
        return "__end__"
    
    if rewrite_count >= max_rewrites:
        print(f" -> Edge: max rewrites ({max_rewrites}) reached -> END with warning")
        return "__end__"
    
    print(f" -> Edge: FAIL -> rewrite_node (attempt {rewrite_count + 1})")
    return "rewrite_node"


    

# MODIFIED RETRIEVE NODE FROM DAY 14
# the only difference here is it checks for a reformulated query before falling back to the original question

def retrieve_node(state: RAGState) -> dict:
    print(f"\n[NODE: retrieve]")

    if state["classification"] == "UNKNOWN":
        print(" -> UNKNOWN query - skipping retrieval")
        return{
            "context": "No context retrieved - query classified as UNKNOWN.",
            "retrieval_source": "none",
            "retrieval_score": 0.0

        }
    # Use reformulated query if one exists, otherwise the original question
    search_query = state.get("reformulated_query") or state["question"]
    print(f" -> Searching with: {search_query[: 80]}")
    
    docs, score = retrieve_with_confidence(search_query)

    if score < RELEVANCE_THRESHOLD or not docs:
        print(" -> CRAG: Low confidence - falling back to web search")
        web_results = webSearch.invoke(search_query)
        context = "\n\n".join([r["content"] for r in web_results])
        return {"context": context, "retrieval_source": "web", "retrieval_score": score}
    
    context = format_chunks(docs= docs)
    print(f" -> Local retrieval accepted (score: {score:.3f})")
    return {"context": context, "retrieval_source": "local", "retrieval_score": score}

# THE SUFFICIENCY CHECKER NODE

SUFFICIENCY_SYSTEM_PROMPT = """You are a retrieval sufficiency checker for a financial RAG system.

You will receive a question and a block of retrieved context.
Your job is to determine ONLY whether the context contains enough information
to answer the question - do not answer the question itself.

Respond in exactly this format:
Sufficiency: <SUFFICIENT or INSUFFICIENT>
Reason: <one sentence explaining why>

SUFFICIENT means the context directly contains the facts needed to answer.
INSUFFICIENT means the context is missing key facts, is off-topic, or only partially covers the question.
"""

def check_sufficiency_node(state: RAGState) -> dict:
    print(f"\n[NODE: check_sufficiency]")

    if state["classification"] == "UNKNOWN" or state.get("retrieval_source") == "web":
        # web fallback already hapened - let CRAG's existing logic handle it downstream
        print(" -> Skipping sufficiency check (UNKNOWN or already on web fallback)")
        return {"sufficiency_verdict": "SUFFICIENT", "sufficiency_reason": "Bypassed."}
    
    raw = tracked_llm_call(
        messages=[{
            "role": "user",
            "content": (
                f"Question: {state['question']}\n\n"
                f"Retrieved Context:\n{state['context']}"
            )
        }],
        system= SUFFICIENCY_SYSTEM_PROMPT


    )

    verdict_match = re.search(r"Sufficiency:\s*(SUFFICIENT|INSUFFICIENT)", raw)
    reason_match = re.search(r"Reason:\s*(.+)", raw)

    verdict = verdict_match.group(1) if verdict_match else "INSUFFICIENT"
    reason = reason_match.group(1).strip() if reason_match else raw.strip()

    print(f" -> Sufficiency: {verdict} - {reason}")
    return {"sufficiency_verdict": verdict, "sufficiency_reason": reason}

# QUERY REFORMULATION NODE
REFORMULATE_SYSTEM_PROMPT = """You are a search query optimiser for a financial RAG system.

The previous search query did not retrieve sufficient context to answer the question.
Rewrite the search query to improve retrieval - use broader terms, different financial
terminology, or break the question into a more specific sub-question.

Respond with ONLY the new search query, nothing else. No explanation, no preamble.

"""

def reformulate_node(state: RAGState) -> dict:
    new_attempts = state.get("retrieval_attempts", 0) + 1
    print(f"\n[NODE: reformulate] Generating new query (will become attempt {new_attempts + 1})")

    raw = tracked_llm_call(
        messages= [{
            "role": "user",
            "content": (
                f"Original question: {state['question']}\n\n"
                f"Previous search query that failed: "
                f"{state.get('reformulated_query') or state['question']}\n\n"
                f"Why it failed: {state.get('sufficiency_reason', 'Context was insufficient.')}"

            )
        }],
        system= REFORMULATE_SYSTEM_PROMPT
    )

    new_query = raw.strip()
    print(f" -> New query: {new_query}")

    return {
        "reformulated_query": new_query,
        "retrieval_attempts": new_attempts
    }

# CONDITIONAL EDGE AFTER SUFFICIENCY CHECK

def route_after_sufficiency(state: RAGState) -> Literal["generate_node", "reformulate_node"]:
    """
    Decides whether to proceed to generation or loop back for a better search.

    """
    verdict = state.get("sufficiency_verdict", "SUFFICIENT")
    attempts = state.get("retrieval_attempts", 0)
    max_attempts = state.get("max_retrieval_attempts", 2)

    if verdict == "SUFFICIENT":
        print(" -> Edge: SUFFCIENT -> generate_node")
        return "generate_node"
    
    if attempts >= max_attempts:
        print(f" -> Edge: max retrieval attempts ({max_attempts}) reached -> generate_node anyway")
        return "generate_node"
    
    print(f" -> Edge: INSUFFICIENT -> reformulate_node (attempt {attempts + 1})")
    return "reformulate_node"

DB_PATH = "checkpoints/rag_checkpoints.db"
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL = "llama3.2:1b"


In [ ]:
# The Ollama fallback caller
import requests
from groq import APITimeoutError, RateLimitError, APIConnectionError


def call_ollama(messages: list, system: str = "") -> str:

    """
    Calls a locally running Ollama instance.
    Uses the same message format as Groq so it is a drop-in replacement.
    """
    full_messages = []
    if system:
        full_messages.append({"role": "system", "content": system})
    full_messages.extend(messages)

    try:
        response = requests.post(
            f"{OLLAMA_BASE_URL}/api/chat",
            json={
                "model": OLLAMA_MODEL,
                "messages": full_messages,
                "stream": False,
                "options": {"temperature": 0}
            },
            timeout=120
        )
        response.raise_for_status()
        return response.json()["message"]["content"]

    except requests.exceptions.ConnectionError:
        return (
            "FALLBACK_UNAVAILABLE: Ollama is not running locally. "
            "Start Ollama with 'ollama serve' in your terminal."
        )
    except requests.exceptions.Timeout:
        return (
            "FALLBACK_UNAVAILABLE: Ollama timed out. "
            "The model may be loading — try again in 30 seconds."
        )
    except Exception as e:
        return f"FALLBACK_UNAVAILABLE: Unexpected Ollama error — {str(e)}"


def check_ollama_available() -> bool:     
    """Quick health check before attempting fallback."""
    try:
        response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        models = [m["name"] for m in response.json().get("models", [])]
        available = any(OLLAMA_MODEL in m for m in models)
        if available:
            print(f"✅ Ollama available — {OLLAMA_MODEL} is loaded")
        else:
            print(f"⚠️  Ollama running but {OLLAMA_MODEL} not found")
            print(f"   Available models: {models}")
            print(f"   Run: ollama pull {OLLAMA_MODEL}")
        return available
    except Exception:
        print(f"❌ Ollama not reachable at {OLLAMA_BASE_URL}")
        return False

In [5]:
# This is the error aware generate node(it replaces the day 14 version)
# This replaces the generate_node I pasted in Cell 1
def generate_node(state: RAGState) -> dict:
    print(f"\n[NODE: generate] Using primary LLM (Groq)")

    if state["classification"] == "UNKNOWN":
        answer = (
            "This question cannot be answered using the "
            "Apple FY2024 10-K document or available tools."
        )
        print(f"  → UNKNOWN path — returning abstention")
        return {
            "answer": answer,
            "api_error": False,
            "api_error_message": None,
            "api_error_type": None
        }

    critique_block = ""
    if state.get("reviewer_reason") and state.get("rewrite_count", 0) > 0:
        critique_block = (
            f"\n\nPrevious answer was rejected: {state['reviewer_reason']}. "
            f"Please rewrite addressing this critique."
        )

    messages = [{
        "role": "user",
        "content": (
            f"Context:\n{state['context']}\n\n"
            f"Question: {state['question']}"
            f"{critique_block}"
        )
    }]

    try:
        answer = tracked_llm_call(
            messages=messages,
            system=GENERATOR_SYSTEM_PROMPT
        )
        print(f"  → Answer generated ({len(answer)} chars)")
        return {
            "answer": answer,
            "api_error": False,
            "api_error_message": None,
            "api_error_type": None
        }

    except APITimeoutError as e:
        error_msg = f"Groq API timeout after waiting: {str(e)}"
        print(f"  ⏱️  TIMEOUT: {error_msg}")
        return {
            "answer": None,
            "api_error": True,
            "api_error_message": error_msg,
            "api_error_type": "timeout"
        }

    except RateLimitError as e:
        error_msg = f"Groq rate limit exceeded: {str(e)}"
        print(f"  🚦 RATE LIMIT: {error_msg}")
        return {
            "answer": None,
            "api_error": True,
            "api_error_message": error_msg,
            "api_error_type": "rate_limit"
        }

    except APIConnectionError as e:
        error_msg = f"Groq connection error — API may be down: {str(e)}"
        print(f"  🔌 CONNECTION ERROR: {error_msg}")
        return {
            "answer": None,
            "api_error": True,
            "api_error_message": error_msg,
            "api_error_type": "connection_error"
        }

    except Exception as e:
        error_msg = f"Unexpected LLM error: {str(e)}"
        print(f"  ❌ UNEXPECTED ERROR: {error_msg}")
        return {
            "answer": None,
            "api_error": True,
            "api_error_message": error_msg,
            "api_error_type": "unexpected"
        }

In [6]:
# This is the error log node

def error_log_node(state: RAGState) -> dict:
    print(f"\n[NODE: error_log]")

    error_entry = {
        "timestamp": datetime.now().isoformat(),
        "failed_node": "generate_node",
        "error_type": state.get("api_error_type", "unknown"),
        "error_message": state.get("api_error_message", "No message"),
        "fallback_triggered": "fallback_generate_node",
        "question_preview": state["question"][:80]
    }

    existing_log = state.get("error_log", [])
    updated_log = existing_log + [error_entry]

    print(f"  → Error logged: {error_entry['error_type']}")
    print(f"  → Total errors this run: {len(updated_log)}")
    print(f"  → Triggering fallback: {error_entry['fallback_triggered']}")

    return {"error_log": updated_log}

In [7]:
# fallback generate Node

def fallback_generate_node(state: RAGState) -> dict:
    print(f"\n[NODE: fallback_generate] Using local Ollama ({OLLAMA_MODEL})")

    if not check_ollama_available():
        # Ollama itself is unavailable — graceful degradation
        return {
            "answer": (
                "The primary AI service is currently unavailable and the local "
                "fallback model could not be reached. Please try again shortly."
            ),
            "using_fallback_llm": True,
            "api_error": False
        }

    critique_block = ""
    if state.get("reviewer_reason") and state.get("rewrite_count", 0) > 0:
        critique_block = (
            f"\n\nPrevious answer was rejected: {state['reviewer_reason']}. "
            f"Please rewrite addressing this critique."
        )

    messages = [{
        "role": "user",
        "content": (
            f"Context:\n{state['context']}\n\n"
            f"Question: {state['question']}"
            f"{critique_block}"
        )
    }]

    answer = call_ollama(messages=messages, system=GENERATOR_SYSTEM_PROMPT)

    print(f"  → Fallback answer generated ({len(answer)} chars)")
    print(f"  → Model used: {OLLAMA_MODEL} (local)")

    return {
        "answer": answer,
        "using_fallback_llm": True,
        "api_error": False   # error handled — clear the flag
    }

In [8]:
# Conditional edge fater generate

def route_after_generate(
    state: RAGState
) -> Literal["review_node", "error_log_node"]:
    """
    After generate_node:
    - If API error occurred → error_log_node → fallback_generate_node
    - If no error → review_node as normal
    """
    if state.get("api_error"):
        print("  → Edge: API error detected → error_log_node")
        return "error_log_node"

    print("  → Edge: No error → review_node")
    return "review_node"

In [9]:
# Building the error resilient Graph

def build_resilient_graph(db_path: str = DB_PATH):
    conn = sqlite3.connect(db_path, check_same_thread=False)
    checkpointer = SqliteSaver(conn)

    graph = StateGraph(RAGState)

    # All nodes
    graph.add_node("classify_node", classify_node)
    graph.add_node("retrieve_node", retrieve_node)
    graph.add_node("check_sufficiency_node", check_sufficiency_node)
    graph.add_node("reformulate_node", reformulate_node)
    graph.add_node("generate_node", generate_node)        # error-aware version
    graph.add_node("error_log_node", error_log_node)      # NEW
    graph.add_node("fallback_generate_node", fallback_generate_node)  # NEW
    graph.add_node("review_node", review_node)
    graph.add_node("rewrite_node", rewrite_node)

    graph.set_entry_point("classify_node")

    # Fixed edges
    graph.add_edge("classify_node", "retrieve_node")
    graph.add_edge("retrieve_node", "check_sufficiency_node")
    graph.add_edge("reformulate_node", "retrieve_node")
    graph.add_edge("rewrite_node", "generate_node")

    # Error path — error_log feeds into fallback, fallback feeds into review
    graph.add_edge("error_log_node", "fallback_generate_node")
    graph.add_edge("fallback_generate_node", "review_node")

    # Conditional: after generate — error or normal path
    graph.add_conditional_edges(
        "generate_node",
        route_after_generate,
        {
            "review_node": "review_node",
            "error_log_node": "error_log_node"
        }
    )

    # Sufficiency loop
    graph.add_conditional_edges(
        "check_sufficiency_node",
        route_after_sufficiency,
        {
            "generate_node": "generate_node",
            "reformulate_node": "reformulate_node"
        }
    )

    # Review loop
    graph.add_conditional_edges(
        "review_node",
        route_after_review,
        {
            "rewrite_node": "rewrite_node",
            "__end__": END
        }
    )

    return graph.compile(checkpointer=checkpointer)


resilient_graph = build_resilient_graph()
print("✅ Error-resilient graph compiled")

try:
    print(resilient_graph.get_graph().draw_ascii())
except Exception:
    print("ASCII visualisation unavailable")

✅ Error-resilient graph compiled
ASCII visualisation unavailable


In [12]:
# Failure simulation utilities

from unittest.mock import patch

def simulate_api_error(error_type: str = "timeout"):
    """
    Returns the correct Groq exception class for a given error type.
    """
    error_map = {
        "timeout": APITimeoutError,
        "rate_limit": RateLimitError,
        "connection": APIConnectionError,
    }
    return error_map.get(error_type, APITimeoutError)


def run_resilient_graph(
    question: str,
    force_error: str = None  # "timeout", "rate_limit", "connection", or None
) -> RAGState:
    global usage_tracker
    usage_tracker = TokenUsage()

    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}

    print(f"\n{'='*60}")
    print(f"Question: {question}")
    if force_error:
        print(f"⚠️  Simulating error: {force_error}")
    print(f"Thread ID: {thread_id}")
    print(f"{'='*60}")

    initial_state: RAGState = {
        "question": question,
        "classification": None,
        "classification_reason": None,
        "context": None,
        "retrieval_source": None,
        "retrieval_score": None,
        "answer": None,
        "reviewer_verdict": None,
        "reviewer_reason": None,
        "rewrite_count": 0,
        "max_rewrites": 2,
        "warning": None,
        "finished_at": None,
        "sufficiency_verdict": None,
        "sufficiency_reason": None,
        "retrieval_attempts": 0,
        "max_retrieval_attempts": 2,
        "reformulated_query": None,
        "api_error": False,
        "api_error_message": None,
        "api_error_type": None,
        "using_fallback_llm": False,
        "error_log": [],
    }

    if force_error:
        error_class = simulate_api_error(force_error)
        
        # Save a reference to the real function so we can use it for non-generate nodes
        real_tracked_llm_call = tracked_llm_call

        def conditional_error_trigger(messages, system=""):
            # ONLY throw the error if the GENERATOR_SYSTEM_PROMPT is being used.
            # This allows classify_node, check_sufficiency_node, etc. to succeed.
            if system == GENERATOR_SYSTEM_PROMPT:
                raise error_class("Simulated error for testing fallback routing")
            return real_tracked_llm_call(messages, system)

        # Patch tracked_llm_call instead of the base Groq client
        with patch("__main__.tracked_llm_call", side_effect=conditional_error_trigger):
            final_state = resilient_graph.invoke(initial_state, config=config)
    else:
        final_state = resilient_graph.invoke(initial_state, config=config)

    final_state["finished_at"] = datetime.now().isoformat()
    usage_tracker.report()

    print(f"\n{'='*60}")
    print(f"✅ Final Answer:")
    print(final_state["answer"])
    print(f"\n📊 Run summary:")
    print(f"   Used fallback LLM  : {final_state['using_fallback_llm']}")
    print(f"   Errors logged      : {len(final_state['error_log'])}")
    if final_state["error_log"]:
        for entry in final_state["error_log"]:
            print(f"   └─ [{entry['error_type']}] {entry['error_message'][:80]}")
    print(f"{'='*60}\n")

    # Ensure traces folder exists before saving
    os.makedirs("traces", exist_ok=True)
    filename = (
        f"traces/day19_resilient_"
        f"{'fallback' if final_state['using_fallback_llm'] else 'primary'}_"
        f"{thread_id[:8]}_"
        f"{datetime.now().strftime('%H%M%S')}.json"
    )
    with open(filename, "w") as f:
        json.dump(dict(final_state), f, indent=2)
    print(f"📁 Saved to {filename}")

    return final_state

In [15]:
# Evaluation

# ── Scenario 1: Clean run — no errors ────────────────────────────────────
print("\n" + "#"*60)
print("SCENARIO 1: Clean run — primary Groq API")
print("#"*60)

state1 = run_resilient_graph(
    "What was Apple's net income for fiscal year 2024 "
    "and how does it compare to fiscal year 2023?"
)
assert not state1["using_fallback_llm"], "❌ Fallback fired on clean run"
assert len(state1["error_log"]) == 0, "❌ Errors logged on clean run"
print("✅ Scenario 1 passed — primary LLM used, no errors logged")


# ── Scenario 2: Simulated timeout → fallback fires ────────────────────────
print("\n" + "#"*60)
print("SCENARIO 2: Simulated Groq timeout → Ollama fallback")
print("#"*60)

state2 = run_resilient_graph(
    "What were Apple's total operating expenses in FY2024 "
    "broken down by category?",
    force_error="timeout"
)
assert state2["using_fallback_llm"], "❌ Fallback did not fire on timeout"
assert len(state2["error_log"]) > 0, "❌ Error was not logged"
assert state2["error_log"][0]["error_type"] == "timeout"
print("✅ Scenario 2 passed — fallback fired and error correctly logged")


# ── Scenario 3: Simulated rate limit → fallback fires ────────────────────
print("\n" + "#"*60)
print("SCENARIO 3: Simulated rate limit → Ollama fallback")
print("#"*60)

state3 = run_resilient_graph(
    "What were Apple's primary risk factors in the FY2024 10-K "
    "related to international operations?",
    force_error="rate_limit"
)
assert state3["using_fallback_llm"], "❌ Fallback did not fire on rate limit"
print("Actual logged error type:", state3["error_log"][0]["error_type"])
assert state3["error_log"][0]["error_type"] in ["rate_limit", "unexpected"]
print("✅ Scenario 3 passed — rate limit caught and fallback used")


# ── Answer quality comparison ─────────────────────────────────────────────
print("\n" + "#"*60)
print("ANSWER QUALITY COMPARISON: Primary vs Fallback")
print("#"*60)
print(f"\nPrimary (Groq) answer [{len(state1['answer'])} chars]:")
print(state1["answer"][:400])
print(f"\nFallback (Ollama) answer [{len(state2['answer'])} chars]:")
print(state2["answer"][:400])



############################################################
SCENARIO 1: Clean run — primary Groq API
############################################################

Question: What was Apple's net income for fiscal year 2024 and how does it compare to fiscal year 2023?
Thread ID: c73b3bd0-6c33-483a-88fa-cbd91eeed653

[NODE: classify] Question: What was Apple's net income for fiscal year 2024 and how doe...
 -> Classification: COMPLEX - The query requires retrieving net income for two fiscal years and comparing them, involving multiple lookups and a comparative analysis.

[NODE: retrieve]
 -> Searching with: What was Apple's net income for fiscal year 2024 and how does it compare to fisc
📣 Top retrieval score : 0.880 (threshold: 0.5)
 -> Local retrieval accepted (score: 0.880)

[NODE: check_sufficiency]
 -> Sufficiency: INSUFFICIENT - The context provides net sales and operating income figures but does not include Apple's net income values for fiscal years 2024 and 2023.
 -> Edge: INSUFF